<a href="https://colab.research.google.com/github/vaishali27-c/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

# Ranked Actions and Reason Codes

## Lane
Refresh / Content Opportunity Scoring

## Ranked Actions

1. Refresh Content
2. Review Metadata
3. Monitor Performance
4. Keep As-Is

## Reason Codes

STALE_CONTENT
LOW_CTR
LOW_IMPRESSIONS
HIGH_PERFORMANCE

In [1]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [2]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HuggingFace,
    TOKEN '{HF_TOKEN}'
)
""")

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM {src}")

con.sql("SHOW TABLES").df()

,name
0,dim_clients
1,dim_content
2,fact_daily
3,fact_daily_sample
4,fact_query_90d


In [6]:
df = con.sql("""
SELECT *
FROM fact_daily_sample
LIMIT 10000
""").df()


In [5]:
import pandas as pd

ranked = df.copy()

# Simple baseline score
ranked["priority_score"] = (
    ranked["gsc_impressions"] * 0.4 +
    ranked["gsc_clicks"] * 0.3 +
    ranked["ga4_sessions"] * 0.2 +
    ranked["scroll_events"] * 0.1
)

ranked["reason_code"] = "STALE_CONTENT"
ranked["recommended_action"] = "Refresh Content"

ranked = ranked.sort_values(
    by="priority_score",
    ascending=False
)

ranked.head(20)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,priority_score,reason_code,recommended_action
2053,2026-06-01,client_3ffa76342f366962,content_8d4c42e5457c9e2e,True,True,True,False,207,0,1442,...,0,0,0,0,0,0,2026-06,82.8,STALE_CONTENT,Refresh Content
1967,2026-06-01,client_3ffa76342f366962,content_e86d749e34db9b73,True,True,True,False,161,0,1220,...,0,0,0,0,0,0,2026-06,64.4,STALE_CONTENT,Refresh Content
2023,2026-06-01,client_3ffa76342f366962,content_ba367d4a53ac36fb,True,True,True,False,81,0,644,...,0,0,0,0,0,0,2026-06,32.4,STALE_CONTENT,Refresh Content
1968,2026-06-01,client_3ffa76342f366962,content_67ae6123bdbf9081,True,True,True,False,71,0,335,...,0,0,0,0,0,0,2026-06,28.4,STALE_CONTENT,Refresh Content
1013,2026-06-01,client_3ffa76342f366962,content_05244c3e432e7b08,True,True,True,False,59,0,406,...,0,0,0,0,0,0,2026-06,23.6,STALE_CONTENT,Refresh Content
1007,2026-06-01,client_3ffa76342f366962,content_b9b011826cf38a50,True,True,True,False,39,0,245,...,0,0,0,0,0,0,2026-06,15.6,STALE_CONTENT,Refresh Content
1189,2026-06-01,client_3ffa76342f366962,content_1e3fe83db5900a14,True,True,True,False,31,0,898,...,0,0,0,0,0,0,2026-06,12.4,STALE_CONTENT,Refresh Content
2926,2026-06-01,client_3ffa76342f366962,content_52c98b3f26951f3b,True,True,True,True,26,2,79,...,0,0,0,0,0,0,2026-06,11.4,STALE_CONTENT,Refresh Content
5435,2026-06-01,client_3ffa76342f366962,content_2142d449720241b3,True,True,True,False,25,0,169,...,0,0,0,0,0,0,2026-06,10.0,STALE_CONTENT,Refresh Content
2029,2026-06-01,client_3ffa76342f366962,content_025a926ce6e87ba5,True,True,True,False,24,0,147,...,0,0,0,0,0,0,2026-06,9.6,STALE_CONTENT,Refresh Content


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended for **content strategists, SEO specialists, and content managers**.

**Intended Use:**

*   **Prioritization of Content Refresh Efforts:** The ranked actions provide a data-driven prioritization of which content pieces to refresh based on their potential impact (defined by the `priority_score`).
*   **Identification of Underperforming Content:** The reason codes help in understanding *why* certain content pieces are flagged for action (e.g., `STALE_CONTENT` in the current baseline).
*   **Decision Support for Content Investment:** By focusing on high-priority content, teams can optimize their efforts and resources for content improvements that are likely to yield the best results.

**Limits:**

*   **Baseline Simplification:** The initial `priority_score` is a simple weighted average and does not account for the nuances of content age, competitive landscape, or seasonal trends. This is a starting point and needs refinement.
*   **Single Reason Code:** Currently, all content is assigned `STALE_CONTENT` as a reason. A more sophisticated model would assign multiple reason codes or more granular reasons based on specific metrics and thresholds.
*   **Lack of Content Context:** The current model ranks based on performance metrics only. It does not incorporate qualitative factors like content type, topical authority, or strategic importance.
*   **Static Thresholds:** The definition of 'stale' or 'low performance' is not dynamically set and might not adapt to evolving market conditions or business goals.
*   **Sample Data Limitations:** The current analysis uses a sample of daily data. Using the full dataset might reveal different trends and priorities.
*   **Future Data Dependence:** The `report_date` is in the future (2026-06-01), indicating that this is a simulated or predictive dataset. Real-world application would require current data.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human review is crucial to ensure the quality and effectiveness of content refresh actions. The following aspects **must be reviewed by a human before acting** on any recommendation:

*   **Content Relevance and Timeliness:** Does the content still align with current business goals, user needs, and market trends? Is the information still accurate and up-to-date?
*   **Strategic Importance:** Is the content part of a core marketing campaign, a foundational resource, or a highly converting piece? Automated scores might not capture its strategic value.
*   **Brand Voice and Tone:** Any refreshed content must maintain the brand's established voice, tone, and editorial guidelines.
*   **Competitive Landscape:** How does the content perform against competitors' similar content? A low `priority_score` might be acceptable if the content serves a niche purpose or performs well relative to its competitive set.
*   **Resource Availability:** Does the team have the resources (writers, designers, subject matter experts) to implement the recommended refresh?
*   **Historical Performance Context:** A human can understand *why* a piece of content might have a low score (e.g., a temporary dip, a new product launch overshadowing older content, seasonal relevance).
*   **User Feedback/Qualitative Data:** Direct feedback from users, sales teams, or support can provide insights that metrics alone cannot.

**What should never be automated:**

*   **Final Content Creation/Rewriting:** While AI can assist with drafting, the final creation, editing, and approval of content must always be human-led to ensure accuracy, brand alignment, and ethical considerations.
*   **Strategic Decision-Making:** The determination of overarching content strategy, target audience, and key messaging requires human insight and judgment.
*   **Ethical Review:** Ensuring content is unbiased, inclusive, and adheres to ethical guidelines is a human responsibility.
*   **Relationship Building (with content):** Understanding the emotional connection or brand loyalty a piece of content might foster is beyond algorithmic capabilities.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations provided by this playbook could go stale if the underlying assumptions or the data characteristics change. Here are triggers that would indicate the need for re-evaluation or retraining:

*   **Significant Shift in Content Performance Metrics:**
    *   A sudden and sustained drop in overall content engagement (e.g., lower average GSC clicks, GA4 sessions, or scroll events) across the board, suggesting the current prioritization model is no longer effective.
    *   A significant change in the distribution of `priority_score` values, either much higher or much lower on average, without a corresponding change in content creation or strategy.

*   **Changes in User Behavior:**
    *   Evolving search trends or user intent, leading to a mismatch between recommended 'refresh' actions and actual content needs.
    *   New platforms or content formats gaining prominence, which might not be adequately captured by current metrics.

*   **Business Strategy or Goals Change:**
    *   A shift in business objectives (e.g., focusing on brand awareness over direct conversions) would require adjusting the weighting of metrics in the `priority_score`.
    *   Introduction of new product lines or target audiences that necessitate a re-evaluation of content relevance.

*   **External Factors:**
    *   Major algorithm updates from search engines (e.g., Google Core Updates) that fundamentally alter how content is ranked and discovered.
    *   Significant changes in the competitive landscape, where competitors might be deploying new content strategies.

*   **Feedback from Human Reviewers:**
    *   Consistent feedback from content strategists or managers that the ranked actions are not intuitive, are missing key content pieces, or are recommending irrelevant actions.
    *   High rates of 'no-go' decisions on recommended actions, indicating a misalignment between the model and human expertise.

*   **Data Source Changes or Degradation:**
    *   Disruption in data pipelines (e.g., GSC or GA4 data becoming unavailable or inconsistent).
    *   Changes in how metrics are collected or defined, which would invalidate historical comparisons and the current scoring model.

**Retraining Triggers:**

When any of the above indicators suggest the recommendations are stale, a retraining or re-evaluation process should be initiated. This would involve:

1.  **Data Refresh:** Updating the underlying `fact_daily` and `fact_query_90d` datasets with the latest available data.
2.  **Metric Review:** Re-evaluating the relevance and weighting of `gsc_impressions`, `gsc_clicks`, `ga4_sessions`, and `scroll_events` (and potentially adding new metrics).
3.  **Model Refinement:** Iterating on the `priority_score` calculation and developing more nuanced `reason_code` logic.
4.  **Threshold Adjustment:** Re-calibrating thresholds for defining 'high-priority' content or specific action categories.
5.  **Validation:** Running the updated playbook against recent historical data to ensure its recommendations align with desired outcomes and expert judgment.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [12]:
import os

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/action_playbook.csv",
    index=False
)

print("Action playbook exported successfully.")

Action playbook exported successfully.


## Self Check

✅ Ranked actions created

✅ Reason codes assigned

✅ Human review included

✅ Monitoring triggers listed

✅ Export created

✅ Decision-support language used